# Jordan-Wide Basin-Aware 3-Model Ensemble — Unified NetCDF Files
## Six Time Periods | SSP2-4.5

**Purpose:**  
Generate **six unified NetCDF files**, each covering the full Jordan domain (85 lat × 75 lon, 0.1° resolution), where every grid cell carries the basin-specific 3-model ensemble precipitation. A `basin_id` coordinate variable is embedded so downstream users can immediately identify which basin each grid cell belongs to.

**Key difference from Notebook 04:**  
Notebook 04 produced one small NC file *per basin* (each cropped to that basin's bounding box). This notebook produces one *Jordan-wide* NC file *per time period*, with all basins embedded. Grid cells outside Jordan are filled with `NaN`.

**Grid cell → basin assignment:**  
Each grid cell centre (lat, lon) is tested with `shapely.Point.within()` against the Jordan basin polygons. The cell is assigned to the basin whose polygon contains its centre. Cells not contained in any Jordan polygon are set to `NaN`.  
This avoids double-assignment at shared borders.

**Ensemble construction per cell:**  
$$P_{\text{ens3}}(i,j,t) = \frac{1}{3}\sum_{k=1}^{3} M_k(i,j,t)$$

where the three models $M_k$ are the top-ranked models for the basin that grid cell $(i,j)$ belongs to.

**Six output files:**

| # | Period | Description |
|---|--------|-------------|
| 1 | 1961–2070 | Full simulation period |
| 2 | 1995–2014 | Evaluation / reference period (IPCC AR6 baseline) |
| 3 | 2015–2020 | Transitional period |
| 4 | 2021–2040 | Near-future period |
| 5 | 2041–2060 | Mid-future period |
| 6 | 2061–2070 | End-of-century period |

**Inputs:**  
- `basin_model_rankings.csv` (Notebook 03)  
- `Jo.shp` — Jordan surface basin shapefile  
- Six model NetCDF files (1961–2070, `prAdjust`, mm/day)

**Output directory:**  
`C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\3 ensemble models\`


## 1. Import Libraries

In [1]:
import os
import warnings
import datetime
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
from pathlib import Path
from scipy.spatial import cKDTree
from shapely.geometry import Point
import time as _time

warnings.filterwarnings("ignore")

print("Libraries loaded.")
for lib, mod in [("numpy", np), ("pandas", pd), ("xarray", xr), ("geopandas", gpd)]:
    print(f"  {lib:<12}: {mod.__version__}")


Libraries loaded.
  numpy       : 2.1.3
  pandas      : 2.2.3
  xarray      : 2025.12.0
  geopandas   : 1.0.1


## 2. Configuration

In [2]:
# ── Input paths ───────────────────────────────────────────────────────────────
SHAPEFILE    = Path(r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments"
                    r"\jordan.basins\surface.basins.for.jordan\Jo.shp")

RANKINGS_CSV = Path(r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments"
                    r"\validation\single.model\basin_model_rankings.csv")

MODEL_DIR    = Path(r"D:\RICAAR\riccar-data_jordan-ssp2-4-5-daily-data_2024-07-29_0915"
                    r"\Merge\Precipitation")

# ── Output directory ──────────────────────────────────────────────────────────
OUTPUT_DIR   = Path(r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments"
                    r"\3 ensemble models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Models ────────────────────────────────────────────────────────────────────
MODELS = [
    "CMCC-CM2-SR5",
    "CNRM-ESM2-1",
    "EC-Earth3-Veg",
    "IPSL-CM6A-LR",
    "MPI-ESM1-2-LR",
    "NorESM2-MM",
]

PR_VAR = "prAdjust"   # precipitation variable in NC files (mm/day)

# ── Syrian basins to exclude from shapefile ───────────────────────────────────
SYRIA_BASINS = {"YARMOUK (SYRIA)", "AZRAQ (SYRIA)", "AMMAN ZARQA (SYRIA)"}

# ── Shapefile BASIN_NAME → validation ranking basin name ─────────────────────
# None = no station data → will inherit from nearest ranked basin via KD-tree
BASIN_NAME_MAP = {
    "HAMMAD"                    : "HAMMAD",
    "YARMOUK (JORDAN)"          : "YARMOUK (JORDAN)",
    "J.VALLEY-YARMOUK TRIANGLE" : None,
    "JORDAN VALLY (JORDAN)"     : "JORDAN VALLY (JORDAN)",
    "N.R.S.W"                   : "N.R.S.W",
    "AZRAQ (JORDAN)"            : "AZRAQ (JORDAN)",
    "AMMAN ZARQA (JORDAN)"      : "AMMAN ZARQA (JORDAN)",
    "S.R.S.W"                   : "S.R.S.W",
    "MUJIB"                     : "MUJIB",
    "D.S.R.S.W"                 : "D.S.R.S.W",
    "W. ARABA NORTH"            : "W. ARABA NORTH",
    "HASA"                      : "HASA",
    "JAFER"                     : "JAFER",
    "WADI ARABA SOUTH"          : None,
    "QA DISI & SOUTHERN DESERT" : None,
    "SARHAN"                    : None,
}

# ── Six time periods ──────────────────────────────────────────────────────────
# (period_label, start_year, end_year)
PERIODS = [
    ("1961_2070", 1961, 2070),   # Full simulation period
    ("1995_2014", 1995, 2014),   # Evaluation / reference period
    ("2015_2020", 2015, 2020),   # Transitional period
    ("2021_2040", 2021, 2040),   # Near-future
    ("2041_2060", 2041, 2060),   # Mid-future
    ("2061_2070", 2061, 2070),   # End-of-century
]

print("Configuration loaded.")
print(f"  Model dir  : {MODEL_DIR}")
print(f"  Shapefile  : {SHAPEFILE}")
print(f"  Output dir : {OUTPUT_DIR}")
print()
print("Six output periods:")
for label, y0, y1 in PERIODS:
    print(f"  {label}  ({y0}–{y1})")


Configuration loaded.
  Model dir  : D:\RICAAR\riccar-data_jordan-ssp2-4-5-daily-data_2024-07-29_0915\Merge\Precipitation
  Shapefile  : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\jordan.basins\surface.basins.for.jordan\Jo.shp
  Output dir : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\3 ensemble models

Six output periods:
  1961_2070  (1961–2070)
  1995_2014  (1995–2014)
  2015_2020  (2015–2020)
  2021_2040  (2021–2040)
  2041_2060  (2041–2060)
  2061_2070  (2061–2070)


## 3. Load Shapefile and Validation Rankings


In [3]:
# ── Shapefile ─────────────────────────────────────────────────────────────────
gdf_all    = gpd.read_file(SHAPEFILE)
gdf_jordan = gdf_all[~gdf_all["BASIN_NAME"].isin(SYRIA_BASINS)].copy().reset_index(drop=True)

print(f"Total shapefile features : {len(gdf_all)}")
print(f"Jordan-only retained     : {len(gdf_jordan)}")
print()
print("Jordan basins:")
for _, row in gdf_jordan.iterrows():
    print(f"  {row['BASIN_NAME']}")

# ── Rankings ──────────────────────────────────────────────────────────────────
rankings = pd.read_csv(RANKINGS_CSV)

# Top-3 per validated basin (sorted by Avg_Rank ascending = best first)
top3_per_basin = {}
for basin, grp in rankings.groupby("Basin"):
    top3_per_basin[basin] = grp.sort_values("Avg_Rank")["Model"].iloc[:3].tolist()

print()
print("Top-3 models per validated basin:")
print(f"{'Basin':<28} {'Rank-1':<22} {'Rank-2':<22} {'Rank-3'}")
print("-" * 88)
for basin, models in top3_per_basin.items():
    print(f"{basin:<28} {'  |  '.join(models)}")


Total shapefile features : 19
Jordan-only retained     : 16

Jordan basins:
  HAMMAD
  YARMOUK (JORDAN)
  J.VALLEY-YARMOUK TRIANGLE
  JORDAN VALLY (JORDAN)
  N.R.S.W
  AZRAQ (JORDAN)
  AMMAN ZARQA (JORDAN)
  S.R.S.W
  MUJIB
  D.S.R.S.W
  W. ARABA NORTH
  HASA
  JAFER
  WADI ARABA SOUTH
  QA DISI & SOUTHERN DESERT
  SARHAN

Top-3 models per validated basin:
Basin                        Rank-1                 Rank-2                 Rank-3
----------------------------------------------------------------------------------------
AMMAN ZARQA (JORDAN)         MPI-ESM1-2-LR  |  NorESM2-MM  |  CMCC-CM2-SR5
AZRAQ (JORDAN)               MPI-ESM1-2-LR  |  NorESM2-MM  |  CMCC-CM2-SR5
D.S.R.S.W                    IPSL-CM6A-LR  |  MPI-ESM1-2-LR  |  NorESM2-MM
HAMMAD                       MPI-ESM1-2-LR  |  CMCC-CM2-SR5  |  NorESM2-MM
HASA                         MPI-ESM1-2-LR  |  CMCC-CM2-SR5  |  CNRM-ESM2-1
JAFER                        NorESM2-MM  |  MPI-ESM1-2-LR  |  CMCC-CM2-SR5
JORDAN VALLY (JORDA

## 4. Assign Top-3 Models to All Jordan Basins

Basins without station-based validation data inherit the top-3 assignment from their nearest validated basin (centroid k-d tree, consistent with Section 2.5 of the manuscript).


In [4]:
gdf_jordan["centroid_lon"] = gdf_jordan.geometry.centroid.x
gdf_jordan["centroid_lat"] = gdf_jordan.geometry.centroid.y

# Separate ranked vs unranked
ranked_mask = gdf_jordan["BASIN_NAME"].apply(
    lambda n: BASIN_NAME_MAP.get(n) is not None
              and BASIN_NAME_MAP.get(n) in top3_per_basin
)
ranked_gdf   = gdf_jordan[ranked_mask].copy()
unranked_gdf = gdf_jordan[~ranked_mask].copy()

# Build KD-tree on ranked centroids
ranked_coords = ranked_gdf[["centroid_lat", "centroid_lon"]].values
tree = cKDTree(ranked_coords)

# Final assignment dict: shapefile BASIN_NAME → {top3, source}
basin_assignment = {}

for _, row in gdf_jordan.iterrows():
    shp_name     = row["BASIN_NAME"]
    ranking_name = BASIN_NAME_MAP.get(shp_name)

    if ranking_name is not None and ranking_name in top3_per_basin:
        top3   = top3_per_basin[ranking_name]
        source = "direct"
    else:
        query        = [row["centroid_lat"], row["centroid_lon"]]
        dist, idx    = tree.query(query)
        nearest_shp  = ranked_gdf.iloc[idx]["BASIN_NAME"]
        nearest_rank = BASIN_NAME_MAP[nearest_shp]
        top3         = top3_per_basin[nearest_rank]
        source       = f"KD-tree -> {nearest_shp} ({dist * 111.32:.1f} km)"

    basin_assignment[shp_name] = {"top3": top3, "source": source}

print(f"{'Basin':<32} {'Assignment source':<38} {'Top-3 Models'}")
print("-" * 105)
for b, info in basin_assignment.items():
    print(f"{b:<32} {info['source']:<38} {' | '.join(info['top3'])}")


Basin                            Assignment source                      Top-3 Models
---------------------------------------------------------------------------------------------------------
HAMMAD                           direct                                 MPI-ESM1-2-LR | CMCC-CM2-SR5 | NorESM2-MM
YARMOUK (JORDAN)                 direct                                 MPI-ESM1-2-LR | NorESM2-MM | CNRM-ESM2-1
J.VALLEY-YARMOUK TRIANGLE        KD-tree -> N.R.S.W (29.2 km)           NorESM2-MM | CNRM-ESM2-1 | CMCC-CM2-SR5
JORDAN VALLY (JORDAN)            direct                                 MPI-ESM1-2-LR | CMCC-CM2-SR5 | NorESM2-MM
N.R.S.W                          direct                                 NorESM2-MM | CNRM-ESM2-1 | CMCC-CM2-SR5
AZRAQ (JORDAN)                   direct                                 MPI-ESM1-2-LR | NorESM2-MM | CMCC-CM2-SR5
AMMAN ZARQA (JORDAN)             direct                                 MPI-ESM1-2-LR | NorESM2-MM | CMCC-CM2-SR5
S.R.S.W         

## 5. Assign Every Jordan-Domain Grid Cell to a Basin

For each of the 85 × 75 = 6,375 grid cell centres in the model domain, `shapely.Point.within()` is used to test containment inside each Jordan basin polygon. Each grid cell is assigned to **exactly one basin** (the one whose polygon contains its centre). Cells not contained in any Jordan polygon receive basin code `-1` (outside Jordan / NaN in output).

This produces two 2-D lookup arrays of shape (85, 75):
- `basin_id_grid` — integer index into the basin list (−1 = outside Jordan)  
- `model_triplet_grid` — list of the 3 model names for that cell's basin


In [5]:
# Load grid coordinates from first model
sample_nc = MODEL_DIR / f"{MODELS[0]}.nc"
with xr.open_dataset(sample_nc) as ds_sample:
    lats       = ds_sample["lat"].values        # (85,)
    lons       = ds_sample["lon"].values        # (75,)
    all_times  = ds_sample["time"].values       # (40177,)
    pr_attrs   = dict(ds_sample[PR_VAR].attrs)
    global_attrs_template = dict(ds_sample.attrs)

n_lat = len(lats)
n_lon = len(lons)
print(f"Model grid: {n_lat} lat × {n_lon} lon = {n_lat * n_lon:,} cells")
print(f"Full time axis: {len(all_times):,} steps  "
      f"({pd.Timestamp(all_times[0]).date()} → {pd.Timestamp(all_times[-1]).date()})")
print()

# Build ordered basin list from Jordan shapefile (preserves shapefile row order)
jordan_basin_names = gdf_jordan["BASIN_NAME"].tolist()
n_basins           = len(jordan_basin_names)

# basin_id_grid: (n_lat, n_lon) → basin index (0-based) or -1
basin_id_grid  = np.full((n_lat, n_lon), -1, dtype=np.int8)

t_start = _time.time()
for li, lat in enumerate(lats):
    for lj, lon in enumerate(lons):
        pt = Point(lon, lat)
        for bi, bname in enumerate(jordan_basin_names):
            if gdf_jordan.iloc[bi].geometry.contains(pt):
                basin_id_grid[li, lj] = bi
                break   # first match wins — no double assignment

elapsed = _time.time() - t_start
n_jordan_cells   = np.sum(basin_id_grid >= 0)
n_outside_cells  = np.sum(basin_id_grid < 0)
print(f"Grid cell classification complete  ({elapsed:.0f}s)")
print(f"  Inside Jordan  : {n_jordan_cells:,} cells")
print(f"  Outside Jordan : {n_outside_cells:,} cells")
print()

# Per-basin cell count
print("Grid cells per basin:")
for bi, bname in enumerate(jordan_basin_names):
    n = int(np.sum(basin_id_grid == bi))
    print(f"  {bname:<32} : {n:>4} cells")


Model grid: 85 lat × 75 lon = 6,375 cells
Full time axis: 40,177 steps  (1961-01-01 → 2070-12-31)

Grid cell classification complete  (3s)
  Inside Jordan  : 840 cells
  Outside Jordan : 5,535 cells

Grid cells per basin:
  HAMMAD                           :  176 cells
  YARMOUK (JORDAN)                 :   11 cells
  J.VALLEY-YARMOUK TRIANGLE        :    0 cells
  JORDAN VALLY (JORDAN)            :    5 cells
  N.R.S.W                          :   11 cells
  AZRAQ (JORDAN)                   :  113 cells
  AMMAN ZARQA (JORDAN)             :   33 cells
  S.R.S.W                          :    6 cells
  MUJIB                            :   61 cells
  D.S.R.S.W                        :   15 cells
  W. ARABA NORTH                   :   27 cells
  HASA                             :   22 cells
  JAFER                            :  116 cells
  WADI ARABA SOUTH                 :   55 cells
  QA DISI & SOUTHERN DESERT        :   36 cells
  SARHAN                           :  153 cells


## 6. Build Per-Cell Model Triplet Lookup

For each model, build a boolean 2-D mask (85 × 75) indicating which grid cells should use that model's data. A cell uses model M if M is in the top-3 list of the basin that cell belongs to. This pre-computation makes the ensemble loop in Cell 8 efficient.


In [6]:
# model_cell_mask[model] = (n_lat, n_lon) bool array
# True where this model contributes to that cell's ensemble
model_cell_mask = {m: np.zeros((n_lat, n_lon), dtype=bool) for m in MODELS}

# Also build a (n_lat, n_lon) array counting how many models contribute per cell
n_models_per_cell = np.zeros((n_lat, n_lon), dtype=np.int8)

for li in range(n_lat):
    for lj in range(n_lon):
        bi = basin_id_grid[li, lj]
        if bi < 0:
            continue
        bname = jordan_basin_names[bi]
        top3  = basin_assignment[bname]["top3"]
        for m in top3:
            model_cell_mask[m][li, lj] = True
        n_models_per_cell[li, lj] = len(top3)

# Sanity check: every Jordan cell should have exactly 3 contributing models
unique_counts = np.unique(n_models_per_cell[basin_id_grid >= 0])
print(f"Unique model-count values for Jordan cells : {unique_counts}")
print(f"  (all should be 3)")
print()

# Summary: how many cells does each model contribute to?
print("Model contribution (number of Jordan grid cells):")
for m in MODELS:
    n = int(model_cell_mask[m].sum())
    print(f"  {m:<22} : {n:>4} cells")


Unique model-count values for Jordan cells : [3]
  (all should be 3)

Model contribution (number of Jordan grid cells):
  CMCC-CM2-SR5           :  814 cells
  CNRM-ESM2-1            :   44 cells
  EC-Earth3-Veg          :    0 cells
  IPSL-CM6A-LR           :   97 cells
  MPI-ESM1-2-LR          :  829 cells
  NorESM2-MM             :  736 cells


## 7. Prepare Basin ID Variable

Each output NetCDF file will include a static 2-D variable `basin_id` (shape: lat × lon, dtype int8) so users can immediately look up which basin any grid cell belongs to. A companion string attribute `basin_id_labels` maps integer codes to basin names.

| Value | Meaning |
|-------|---------|
| 0 … N-1 | Jordan basin index (see `basin_id_labels` attribute) |
| -1 | Outside Jordan domain (NaN in precipitation variable) |


In [7]:
# basin_id_grid already built in Cell 5  (int8, -1 = outside)
# Build a human-readable label string
basin_id_labels = "; ".join(
    f"{bi}={bname}" for bi, bname in enumerate(jordan_basin_names)
)
print("basin_id encoding:")
for bi, bname in enumerate(jordan_basin_names):
    n_cells = int(np.sum(basin_id_grid == bi))
    top3    = basin_assignment[bname]["top3"]
    print(f"  {bi:>2} = {bname:<32}  cells={n_cells:>4}  models={' | '.join(top3)}")
print()
print(f" -1 = outside Jordan domain  (cells={int(np.sum(basin_id_grid < 0)):,})")


basin_id encoding:
   0 = HAMMAD                            cells= 176  models=MPI-ESM1-2-LR | CMCC-CM2-SR5 | NorESM2-MM
   1 = YARMOUK (JORDAN)                  cells=  11  models=MPI-ESM1-2-LR | NorESM2-MM | CNRM-ESM2-1
   2 = J.VALLEY-YARMOUK TRIANGLE         cells=   0  models=NorESM2-MM | CNRM-ESM2-1 | CMCC-CM2-SR5
   3 = JORDAN VALLY (JORDAN)             cells=   5  models=MPI-ESM1-2-LR | CMCC-CM2-SR5 | NorESM2-MM
   4 = N.R.S.W                           cells=  11  models=NorESM2-MM | CNRM-ESM2-1 | CMCC-CM2-SR5
   5 = AZRAQ (JORDAN)                    cells= 113  models=MPI-ESM1-2-LR | NorESM2-MM | CMCC-CM2-SR5
   6 = AMMAN ZARQA (JORDAN)              cells=  33  models=MPI-ESM1-2-LR | NorESM2-MM | CMCC-CM2-SR5
   7 = S.R.S.W                           cells=   6  models=MPI-ESM1-2-LR | NorESM2-MM | CMCC-CM2-SR5
   8 = MUJIB                             cells=  61  models=MPI-ESM1-2-LR | NorESM2-MM | CMCC-CM2-SR5
   9 = D.S.R.S.W                         cells=  15  models=IPSL-CM6

## 8. Generate the Six Unified Jordan-Wide NetCDF Files

**Processing strategy (memory-efficient):**  
Rather than loading all six model arrays into RAM simultaneously, the loop opens each model file lazily, extracts only the time slice needed, and accumulates the ensemble sum cell-by-cell using pre-built boolean masks. Peak memory per period = 3 model arrays × (n_time_slice × n_lat × n_lon × 4 bytes).

For the full period (40,177 steps × 85 × 75 × 4 bytes × 3 models ≈ 3.1 GB) the loop processes each model's contribution sequentially and releases it before opening the next, keeping peak RAM to ≈ 1 model-period array at a time.

**Output filename convention:**  
`Jordan_3model_ensemble_ssp245_<PERIOD_LABEL>.nc`  
e.g. `Jordan_3model_ensemble_ssp245_1995_2014.nc`


In [8]:
all_times_pd = pd.DatetimeIndex(all_times)

for period_label, y_start, y_end in PERIODS:
    out_filename = f"Jordan_3model_ensemble_ssp245_{period_label}.nc"
    out_path     = OUTPUT_DIR / out_filename

    if out_path.exists():
        print(f"[SKIP] {out_filename} already exists.")
        continue

    # ── Select time indices for this period ───────────────────────────────────
    time_mask   = (all_times_pd.year >= y_start) & (all_times_pd.year <= y_end)
    time_idxs   = np.where(time_mask)[0]
    period_times= all_times[time_idxs]
    n_t         = len(time_idxs)

    print(f"[PROCESSING] {out_filename}")
    print(f"  Period    : {y_start}–{y_end}  ({n_t:,} time steps)")
    t0 = _time.time()

    # ── Allocate ensemble accumulator ─────────────────────────────────────────
    # Shape: (n_t, n_lat, n_lon)  — float32 to save memory
    ens_sum = np.zeros((n_t, n_lat, n_lon), dtype=np.float32)
    # NaN mask: cells outside Jordan start as NaN
    outside_mask = (basin_id_grid < 0)   # shape (n_lat, n_lon)

    # ── Accumulate each model's contribution ──────────────────────────────────
    for model in MODELS:
        cell_mask = model_cell_mask[model]   # (n_lat, n_lon) bool
        if not cell_mask.any():
            continue

        nc_path = MODEL_DIR / f"{model}.nc"
        with xr.open_dataset(nc_path) as ds_m:
            # Load only required time steps — still lazy until .values
            pr_model = ds_m[PR_VAR].isel(time=time_idxs).values  # (n_t, n_lat, n_lon)

        # Add to accumulator only where this model is assigned
        # Broadcast mask across time dimension
        ens_sum[:, cell_mask] += pr_model[:, cell_mask]
        del pr_model   # free memory immediately
        print(f"  {model:<22} accumulated")

    # ── Divide by 3 for cells with data, NaN elsewhere ────────────────────────
    ens_pr = np.where(
        outside_mask[np.newaxis, :, :],
        np.nan,
        ens_sum / 3.0
    ).astype(np.float32)
    del ens_sum

    # ── Build xarray Dataset ──────────────────────────────────────────────────
    pr_da = xr.DataArray(
        ens_pr,
        dims=["time", "lat", "lon"],
        coords={"time": period_times, "lat": lats, "lon": lons},
        name=PR_VAR,
        attrs={
            "standard_name" : "precipitation_flux",
            "long_name"     : "Basin-Specific 3-Model Ensemble Mean Precipitation",
            "units"         : "mm/day",
            "cell_methods"  : "time: mean",
            "comment"       : ("Each grid cell uses the arithmetic mean of the "
                               "top-3 ranked CMIP6 models for its basin. "
                               "See basin_id variable for basin assignment. "
                               "Grid cells outside Jordan are NaN."),
        }
    )

    basin_id_da = xr.DataArray(
        basin_id_grid,
        dims=["lat", "lon"],
        coords={"lat": lats, "lon": lons},
        name="basin_id",
        attrs={
            "long_name"        : "Jordan hydrological basin index",
            "flag_values"      : list(range(len(jordan_basin_names))) + [-1],
            "flag_meanings"    : " ".join(
                [b.replace(" ", "_").replace(".", "_").replace("(", "").replace(")", "")
                 for b in jordan_basin_names]
            ) + " outside_jordan",
            "basin_id_labels"  : basin_id_labels,
            "note"             : "-1 = outside Jordan domain",
        }
    )

    # Build per-basin model assignment attribute string
    model_assignment_str = " || ".join(
        f"{b}: {' + '.join(basin_assignment[b]['top3'])}"
        for b in jordan_basin_names
    )

    ds_out = xr.Dataset(
        {PR_VAR: pr_da, "basin_id": basin_id_da},
        attrs={
            "title"              : (f"Jordan-Wide Basin-Specific 3-Model Ensemble "
                                    f"Precipitation {y_start}-{y_end}"),
            "institution"        : "Jordan University of Science and Technology",
            "scenario"           : "SSP2-4.5",
            "period"             : f"{y_start}-{y_end}",
            "domain"             : "Jordan  (0.1deg x 0.1deg)",
            "original_data"      : ("RICCAR SMHI-HCLIM-ALADIN-38, "
                                    "bias-corrected via SMHI-MIdAS021"),
            "ensemble_method"    : ("Basin-specific top-3 model ensemble. "
                                    "Basin assignment by centroid-in-polygon. "
                                    "Unranked basins assigned via centroid KD-tree."),
            "basin_model_assignments": model_assignment_str,
            "Conventions"        : "CF-1.7",
            "time_coverage_start": str(pd.Timestamp(period_times[0]).date()),
            "time_coverage_end"  : str(pd.Timestamp(period_times[-1]).date()),
            "history"            : (
                f"Created {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | "
                f"Basin ranking from Hussien et al. composite evaluation framework | "
                f"3-model ensemble: arithmetic mean of top-3 per basin"
            ),
        }
    )

    # Coordinate attributes
    ds_out["lat"].attrs  = {"units": "degrees_north", "long_name": "latitude",  "axis": "Y"}
    ds_out["lon"].attrs  = {"units": "degrees_east",  "long_name": "longitude", "axis": "X"}
    ds_out["time"].attrs = {"long_name": "time"}

    # ── Write compressed NetCDF4 ──────────────────────────────────────────────
    encoding = {
        PR_VAR: {
            "dtype"      : "float32",
            "zlib"       : True,
            "complevel"  : 4,
            "_FillValue" : np.float32(1e+20),
        },
        "basin_id": {
            "dtype"      : "int8",
            "zlib"       : True,
            "complevel"  : 4,
            "_FillValue" : np.int8(-1),
        },
    }

    ds_out.to_netcdf(out_path, encoding=encoding, format="NETCDF4")
    del ens_pr, ds_out

    elapsed = _time.time() - t0
    size_mb = out_path.stat().st_size / 1024 / 1024
    print(f"  Saved     : {out_filename}  ({size_mb:.1f} MB)  [{elapsed:.0f}s]")
    print()

print("=" * 65)
print("ALL SIX FILES COMPLETE")
print("=" * 65)


[PROCESSING] Jordan_3model_ensemble_ssp245_1961_2070.nc
  Period    : 1961–2070  (40,177 time steps)
  CMCC-CM2-SR5           accumulated
  CNRM-ESM2-1            accumulated
  IPSL-CM6A-LR           accumulated
  MPI-ESM1-2-LR          accumulated
  NorESM2-MM             accumulated
  Saved     : Jordan_3model_ensemble_ssp245_1961_2070.nc  (89.4 MB)  [16s]

[PROCESSING] Jordan_3model_ensemble_ssp245_1995_2014.nc
  Period    : 1995–2014  (7,305 time steps)
  CMCC-CM2-SR5           accumulated
  CNRM-ESM2-1            accumulated
  IPSL-CM6A-LR           accumulated
  MPI-ESM1-2-LR          accumulated
  NorESM2-MM             accumulated
  Saved     : Jordan_3model_ensemble_ssp245_1995_2014.nc  (18.0 MB)  [3s]

[PROCESSING] Jordan_3model_ensemble_ssp245_2015_2020.nc
  Period    : 2015–2020  (2,192 time steps)
  CMCC-CM2-SR5           accumulated
  CNRM-ESM2-1            accumulated
  IPSL-CM6A-LR           accumulated
  MPI-ESM1-2-LR          accumulated
  NorESM2-MM             accum

## 9. Verify Output Files

Inspect all six output files: dimensions, time range, basin_id encoding, precipitation statistics, and file size.


In [9]:
nc_files = sorted(OUTPUT_DIR.glob("Jordan_3model_ensemble_ssp245_*.nc"))
print(f"Unified NC files found : {len(nc_files)}")
print()

for nc_f in nc_files:
    with xr.open_dataset(nc_f) as ds_v:
        pr   = ds_v[PR_VAR]
        bid  = ds_v["basin_id"]
        n_t  = len(ds_v["time"])
        t0   = pd.Timestamp(ds_v["time"].values[0]).date()
        t1   = pd.Timestamp(ds_v["time"].values[-1]).date()

        print(f"{'='*65}")
        print(f"File       : {nc_f.name}")
        print(f"Period     : {t0} → {t1}  ({n_t:,} steps)")
        print(f"Grid       : {len(ds_v.lat)} lat × {len(ds_v.lon)} lon")
        print(f"pr range   : {float(pr.min()):.4f} – {float(pr.max()):.4f} mm/day")
        print(f"pr mean    : {float(pr.mean()):.4f} mm/day  (Jordan cells only)")

        # Basin cell counts from basin_id
        bid_np = bid.values
        print(f"basin_id   : unique values = {np.unique(bid_np[bid_np >= 0]).tolist()}")
        print(f"Outside JO : {int(np.sum(bid_np < 0)):,} cells  | "
              f"Inside JO : {int(np.sum(bid_np >= 0)):,} cells")
        print(f"File size  : {nc_f.stat().st_size/1024/1024:.1f} MB")
print()


Unified NC files found : 6

File       : Jordan_3model_ensemble_ssp245_1961_2070.nc
Period     : 1961-01-01 → 2070-12-31  (40,177 steps)
Grid       : 85 lat × 75 lon
pr range   : 0.0000 – 71.2482 mm/day
pr mean    : 0.3118 mm/day  (Jordan cells only)
basin_id   : unique values = [0.0, 1.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0]
Outside JO : 0 cells  | Inside JO : 840 cells
File size  : 89.4 MB
File       : Jordan_3model_ensemble_ssp245_1995_2014.nc
Period     : 1995-01-01 → 2014-12-31  (7,305 steps)
Grid       : 85 lat × 75 lon
pr range   : 0.0000 – 41.9583 mm/day
pr mean    : 0.2866 mm/day  (Jordan cells only)
basin_id   : unique values = [0.0, 1.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0]
Outside JO : 0 cells  | Inside JO : 840 cells
File size  : 18.0 MB
File       : Jordan_3model_ensemble_ssp245_2015_2020.nc
Period     : 2015-01-01 → 2020-12-31  (2,192 steps)
Grid       : 85 lat × 75 lon
pr range   : 0.0000 – 71.2482 mm/d

## 10. Summary

Final output file listing with basin-model assignments embedded as a reference table for reporting.


In [10]:
# ── File listing ──────────────────────────────────────────────────────────────
print("OUTPUT FILES")
print("=" * 75)
total_mb = 0
for f in sorted(OUTPUT_DIR.glob("Jordan_3model_ensemble_ssp245_*.nc")):
    mb = f.stat().st_size / 1024 / 1024
    total_mb += mb
    print(f"  {f.name:<55}  {mb:>7.1f} MB")
print(f"  {'TOTAL':55}  {total_mb:>7.1f} MB")

print()
print("BASIN  →  TOP-3 MODEL ASSIGNMENT")
print("=" * 75)
print(f"{'Basin':<32} {'Assign method':<30} {'Top-3 Models'}")
print("-" * 90)
for bname in jordan_basin_names:
    info = basin_assignment[bname]
    cells_n = int(np.sum(basin_id_grid == jordan_basin_names.index(bname)))
    print(f"{bname:<32} {info['source']:<30} {' | '.join(info['top3'])}")

# ── Save summary CSV ──────────────────────────────────────────────────────────
summary_rows = []
for bi, bname in enumerate(jordan_basin_names):
    info    = basin_assignment[bname]
    top3    = info["top3"]
    cells_n = int(np.sum(basin_id_grid == bi))
    summary_rows.append({
        "basin_id"       : bi,
        "Basin"          : bname,
        "N_Grid_Cells"   : cells_n,
        "Assignment"     : info["source"],
        "Rank1_Model"    : top3[0] if len(top3) > 0 else "-",
        "Rank2_Model"    : top3[1] if len(top3) > 1 else "-",
        "Rank3_Model"    : top3[2] if len(top3) > 2 else "-",
    })

summary_df = pd.DataFrame(summary_rows)
summary_csv = OUTPUT_DIR / "Jordan_ensemble_basin_assignments.csv"
summary_df.to_csv(summary_csv, index=False)
print()
print(f"Basin assignment table saved: {summary_csv}")


OUTPUT FILES
  Jordan_3model_ensemble_ssp245_1961_2070.nc                  89.4 MB
  Jordan_3model_ensemble_ssp245_1995_2014.nc                  18.0 MB
  Jordan_3model_ensemble_ssp245_2015_2020.nc                   5.5 MB
  Jordan_3model_ensemble_ssp245_2021_2040.nc                  18.0 MB
  Jordan_3model_ensemble_ssp245_2041_2060.nc                  18.0 MB
  Jordan_3model_ensemble_ssp245_2061_2070.nc                   9.1 MB
  TOTAL                                                      158.1 MB

BASIN  →  TOP-3 MODEL ASSIGNMENT
Basin                            Assign method                  Top-3 Models
------------------------------------------------------------------------------------------
HAMMAD                           direct                         MPI-ESM1-2-LR | CMCC-CM2-SR5 | NorESM2-MM
YARMOUK (JORDAN)                 direct                         MPI-ESM1-2-LR | NorESM2-MM | CNRM-ESM2-1
J.VALLEY-YARMOUK TRIANGLE        KD-tree -> N.R.S.W (29.2 km)   NorESM2-MM | CNRM-ES